# ELA + CNN Image Manipulation Detection

Notebook Google Colab untuk binary classification gambar **REAL/FAKE** dengan Error Level Analysis (ELA), CNN, evaluasi accuracy + confusion matrix, dan GUI Gradio.

Struktur dataset yang didukung:

```text
dataset/
  real/        # atau authentic/original/asli
  fake/        # atau tampered/manipulated/manipulasi
```

Atau split eksplisit:

```text
dataset/
  train/real
  train/fake
  val/real
  val/fake
```

JPEG dan PNG didukung. PNG/non-JPEG dikonversi ke JPEG canonical sebelum ELA agar efek recompression konsisten. Gambar resolusi rendah dinaikkan minimal ke `MIN_SIDE` sebelum ELA.

In [ ]:
!nvidia-smi || true
!pip -q install tensorflow gradio scikit-learn matplotlib pillow numpy

## 1. Upload / mount dataset

Pilih salah satu: upload zip dataset, mount Google Drive, atau isi `DATA_DIR` langsung ke folder dataset.

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # upload dataset.zip, atau skip cell ini jika pakai Drive
for name in uploaded:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as zf:
            zf.extractall('/content/input_dataset')

# Ubah jika struktur zip punya root folder berbeda.
DATA_DIR = Path('/content/input_dataset')

In [ ]:
# Alternatif Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/path/to/dataset')

## 2. Konfigurasi

In [ ]:
IMAGE_SIZE = 224
MIN_SIDE = 224
SOURCE_JPEG_QUALITY = 95
ELA_RECOMPRESS_QUALITY = 90
ELA_SCALE = 12.0
BATCH_SIZE = 32
EPOCHS = 12
VAL_SPLIT = 0.2
USE_PRETRAINED = True   # EfficientNetB0; set False untuk CNN kecil dari awal
USE_AUGMENTATION = True
ELA_DATASET_DIR = Path('/content/ela_dataset')
MODEL_PATH = Path('/content/ela_cnn_model.keras')
REPORTS_DIR = Path('/content/ela_cnn_reports')

## 3. Utility ELA dan dataset preparation

In [ ]:
import io, json, random, shutil
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
REAL_ALIASES = {'real', 'authentic', 'original', 'asli', 'genuine', '00_real'}
FAKE_ALIASES = {'fake', 'tampered', 'manipulated', 'manipulasi', 'forged', '01_fake'}
OUTPUT_CLASS_DIRS = {'REAL': '00_real', 'FAKE': '01_fake'}

@dataclass(frozen=True)
class ElaConfig:
    image_size: int = IMAGE_SIZE
    min_side: int = MIN_SIDE
    source_jpeg_quality: int = SOURCE_JPEG_QUALITY
    ela_recompress_quality: int = ELA_RECOMPRESS_QUALITY
    ela_scale: float = ELA_SCALE
    seed: int = 42

@dataclass(frozen=True)
class TrainConfig:
    epochs: int = EPOCHS
    batch_size: int = BATCH_SIZE
    learning_rate: float = 1e-4
    val_split: float = VAL_SPLIT
    use_pretrained: bool = USE_PRETRAINED
    use_augmentation: bool = USE_AUGMENTATION
    patience: int = 4

def display_class_name(class_name):
    return 'REAL' if class_name.endswith('real') else 'FAKE' if class_name.endswith('fake') else class_name.upper()

def list_supported_images(directory):
    return [p for p in sorted(Path(directory).rglob('*')) if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]

def load_rgb_image(path):
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert('RGB')

def ensure_min_resolution(image, min_side):
    width, height = image.size
    current_min = min(width, height)
    if current_min >= min_side:
        return image
    scale = min_side / float(current_min)
    return image.resize((int(round(width * scale)), int(round(height * scale))), Image.Resampling.LANCZOS)

def jpeg_roundtrip(image, quality):
    buffer = io.BytesIO()
    image.save(buffer, format='JPEG', quality=quality, subsampling=0, optimize=False)
    buffer.seek(0)
    return Image.open(buffer).convert('RGB')

def compute_ela_image(image, config):
    image = ensure_min_resolution(image.convert('RGB'), config.min_side)
    canonical = jpeg_roundtrip(image, config.source_jpeg_quality)  # handles PNG/non-JPEG consistently
    recompressed = jpeg_roundtrip(canonical, config.ela_recompress_quality)
    arr_a = np.asarray(canonical, dtype=np.int16)
    arr_b = np.asarray(recompressed, dtype=np.int16)
    diff = np.abs(arr_a - arr_b).astype(np.float32) * float(config.ela_scale)
    return Image.fromarray(np.clip(diff, 0, 255).astype(np.uint8), mode='RGB')

def normalise_name(path):
    return Path(path).name.lower().replace('-', '_').replace(' ', '_')

def class_for_dir(path):
    name = normalise_name(path)
    if name in REAL_ALIASES:
        return 'REAL'
    if name in FAKE_ALIASES:
        return 'FAKE'
    return None

def direct_class_dirs(container):
    out = {'REAL': [], 'FAKE': []}
    for child in sorted(Path(container).iterdir()):
        if child.is_dir():
            cls = class_for_dir(child)
            if cls:
                out[cls].append(child)
    return out

def discover_dataset(data_dir):
    data_dir = Path(data_dir).resolve()
    aliases = {'train': 'train', 'training': 'train', 'val': 'val', 'valid': 'val', 'validation': 'val', 'test': 'test'}
    discovered = {}
    for child in sorted(data_dir.iterdir()):
        if child.is_dir() and normalise_name(child) in aliases:
            split = aliases[normalise_name(child)]
            dirs = direct_class_dirs(child)
            if dirs['REAL'] and dirs['FAKE']:
                discovered[split] = dirs
    if discovered:
        return discovered
    root_dirs = direct_class_dirs(data_dir)
    if root_dirs['REAL'] and root_dirs['FAKE']:
        return {'all': root_dirs}
    raise ValueError(f'Tidak menemukan folder class real/fake di {data_dir}')

def split_paths(paths, val_split, seed):
    rng = random.Random(seed)
    shuffled = list(paths)
    rng.shuffle(shuffled)
    val_count = max(1, int(round(len(shuffled) * val_split))) if len(shuffled) > 1 else 0
    return shuffled[val_count:], shuffled[:val_count]

def copy_as_ela(paths, out_dir, config):
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, source_path in enumerate(paths):
        ela = compute_ela_image(load_rgb_image(source_path), config)
        ela.save(out_dir / f'{source_path.stem}_{i:05d}.png', format='PNG')
    return len(paths)

def prepare_ela_dataset(data_dir, out_dir, ela_config, train_config):
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    discovered = discover_dataset(data_dir)
    plan = {'train': {'REAL': [], 'FAKE': []}, 'val': {'REAL': [], 'FAKE': []}}
    if 'all' in discovered:
        for cls in ('REAL', 'FAKE'):
            paths = []
            for class_dir in discovered['all'][cls]:
                paths.extend(list_supported_images(class_dir))
            train_paths, val_paths = split_paths(paths, train_config.val_split, ela_config.seed)
            plan['train'][cls], plan['val'][cls] = train_paths, val_paths
    else:
        for split in ('train', 'val'):
            for cls in ('REAL', 'FAKE'):
                for class_dir in discovered.get(split, {'REAL': [], 'FAKE': []})[cls]:
                    plan[split][cls].extend(list_supported_images(class_dir))
    counts = {'train': {}, 'val': {}}
    for split in ('train', 'val'):
        for cls in ('REAL', 'FAKE'):
            counts[split][cls] = copy_as_ela(plan[split][cls], out_dir / split / OUTPUT_CLASS_DIRS[cls], ela_config)
            if counts[split][cls] == 0:
                raise ValueError(f'Sample kosong untuk {split}/{cls}')
    (out_dir / 'metadata.json').write_text(json.dumps({'ela_config': asdict(ela_config), 'train_config': asdict(train_config), 'counts': counts}, indent=2))
    return counts

## 4. Generate ELA dataset

In [ ]:
ela_config = ElaConfig()
train_config = TrainConfig()
counts = prepare_ela_dataset(DATA_DIR, ELA_DATASET_DIR, ela_config, train_config)
counts

In [ ]:
import matplotlib.pyplot as plt
sample_path = next((ELA_DATASET_DIR / 'train').rglob('*.png'))
plt.figure(figsize=(5, 5))
plt.imshow(Image.open(sample_path))
plt.axis('off')
plt.title(f'ELA preview: {sample_path.parent.name}')

## 5. Train CNN binary classifier

In [ ]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    ELA_DATASET_DIR / 'train', labels='inferred', label_mode='binary',
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
    shuffle=True, seed=ela_config.seed,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    ELA_DATASET_DIR / 'val', labels='inferred', label_mode='binary',
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
    shuffle=False,
)
class_names = list(train_ds.class_names)
class_names, [display_class_name(c) for c in class_names]

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = inputs
if USE_AUGMENTATION:
    x = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.04),
        tf.keras.layers.RandomZoom(0.08),
        tf.keras.layers.RandomContrast(0.12),
    ], name='augmentation')(x)

if USE_PRETRAINED:
    base = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    base.trainable = False
    x = tf.keras.applications.efficientnet.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
else:
    x = tf.keras.layers.Rescaling(1./255)(x)
    for filters in (32, 64, 128):
        x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.35)(x)

outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs, name='ela_cnn_binary_classifier')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=train_config.patience, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
model.save(MODEL_PATH)
Path('/content/class_names.json').write_text(json.dumps(class_names, indent=2))

## 6. Evaluasi: accuracy + confusion matrix

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
y_true, y_prob = [], []
for batch_images, batch_labels in val_ds:
    probs = model.predict(batch_images, verbose=0).reshape(-1)
    y_prob.extend(probs.tolist())
    y_true.extend(np.asarray(batch_labels).reshape(-1).astype(int).tolist())
y_pred = (np.asarray(y_prob) >= 0.5).astype(int)
accuracy = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
labels = [display_class_name(c) for c in class_names]
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(cmap='Blues', values_format='d')
plt.title(f'ELA-CNN Confusion Matrix (accuracy={accuracy:.3f})')
plt.savefig(REPORTS_DIR / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()
metrics = {'accuracy': float(accuracy), 'confusion_matrix': cm.tolist(), 'class_names': class_names}
(REPORTS_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
metrics

## 7. Gradio GUI: upload gambar, preview ELA, output prediksi

In [ ]:
import gradio as gr

def predict_image(image):
    if image is None:
        return None, 'Upload image first.', {}
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
    image = image.convert('RGB')
    ela = compute_ela_image(image, ela_config)
    arr = np.asarray(ela.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR), dtype=np.float32)[None, ...]
    prob_fake = float(model.predict(arr, verbose=0).reshape(-1)[0])
    idx = 1 if prob_fake >= 0.5 else 0
    label = display_class_name(class_names[idx])
    confidence = prob_fake if idx == 1 else 1.0 - prob_fake
    probabilities = {display_class_name(class_names[0]): 1.0 - prob_fake, display_class_name(class_names[1]): prob_fake}
    return ela, f'{label} ({confidence:.2%})', probabilities

demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type='pil', label='Upload image (JPEG/PNG/low-res supported)'),
    outputs=[gr.Image(type='pil', label='ELA preview'), gr.Textbox(label='Prediction'), gr.Label(label='Confidence')],
    title='ELA + CNN Image Manipulation Detector',
    description='Upload gambar untuk melihat preview ELA dan prediksi REAL/FAKE + confidence.',
)
demo.launch(share=True, debug=True)